In [1]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import numpy as np

import ModelFxp12 as fxp

In [2]:
# Create Transforms
transform = transforms.Compose([
    transforms.ToTensor()
])

# Create Datasets
trainset = torchvision.datasets.MNIST(root='MNIST', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='MNIST', train=False, download=True, transform=transform)

# Load Datasets
train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 9912422/9912422 [00:00<00:00, 109785057.99it/s]


Extracting MNIST/MNIST/raw/train-images-idx3-ubyte.gz to MNIST/MNIST/raw



100%|██████████| 28881/28881 [00:00<00:00, 78202513.77it/s]


Extracting MNIST/MNIST/raw/train-labels-idx1-ubyte.gz to MNIST/MNIST/raw



100%|██████████| 1648877/1648877 [00:00<00:00, 28374983.16it/s]


Extracting MNIST/MNIST/raw/t10k-images-idx3-ubyte.gz to MNIST/MNIST/raw



100%|██████████| 4542/4542 [00:00<00:00, 13021550.76it/s]


Extracting MNIST/MNIST/raw/t10k-labels-idx1-ubyte.gz to MNIST/MNIST/raw



In [3]:
class NetMNIST(torch.nn.Module):
    def __init__(self, num_classes = 3):
        torch.nn.Module.__init__(self)
        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3),
            torch.nn.MaxPool2d(2,2),
            torch.nn.ReLU()
        )
        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=4, out_channels=4, kernel_size=3),
            torch.nn.MaxPool2d(2,2),
            torch.nn.ReLU()
        )
        self.fc1 = torch.nn.Linear(in_features=4 * 5 * 5, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        #print(x.shape)
        x = x.view(x.size()[0], -1)
        x = self.fc1(x)
        return x

In [4]:
# Computes the average accuracy
def evaluate_fxp(model, val_loader):

    batch_accuracies = []   # Accuracies of each forwarded batch

    for images, labels in val_loader:
        outputs = model(images)                    # Generates batch predictions

        # Computes the batch accuracy
        _, preds = torch.max(torch.from_numpy(outputs), dim=1)
        accuracy = torch.tensor(torch.sum(preds == labels).item() / len(preds))

        # Append the batch accuracy
        batch_accuracies.append(accuracy)

    # Computes the epoch average accuracy
    accuracy = torch.stack(batch_accuracies).mean() # Average accuracy

    return accuracy

In [5]:
# Create the fixed point model
def model_fxp(model, shift):

    layers = [
        # conv1
        fxp.Conv(model.conv1[0], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv2
        fxp.Conv(model.conv2[0], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # fc
        fxp.Linear(model.fc1, shift, shift)
    ]

    return fxp.ModelFxp(layers)

In [6]:
# Load the quantized floating point model
model = NetMNIST()
model.load_state_dict(torch.load('train2_qat_sh7-9801_fxp-4f.pt', map_location=torch.device('cpu')))

# Report (GetChannel(1))
for shift in (range(7,8)):
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_fxp(model, shift), test_loader)}\n")

#     Parameters          shift   accuracy
#  train2_qat_sh3.pt        3       97.67
#  train2_qat_sh3.pt        4       98.29
#  train2_qat_sh3.pt        5       98.46
#  train2_qat_sh3.pt        6       98.51
#  train2_qat_sh3.pt        7       98.53
#  train2_qat_sh3.pt        8       98.56

#  train2_qat_sh4.pt        4       98.97
#  train2_qat_sh4.pt        5       99.12
#  train2_qat_sh4.pt        6       99.09
#  train2_qat_sh4.pt
#  train2_qat_sh4.pt
#  train2_qat_sh4.pt

#  train2_qat_sh5.pt        5       99.08
#  train2_qat_sh5.pt        6       9910
#  train2_qat_sh5.pt
#  train2_qat_sh5.pt
#  train2_qat_sh5.pt

#  train2_qat_sh6.pt
#  train2_qat_sh6.pt
#  train2_qat_sh6.pt
#  train2_qat_sh6.pt

#  train2_qat_sh7.pt
#  train2_qat_sh7.pt
#  train2_qat_sh7.pt


layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear
Shift: 7, accuracy: 0.9801226258277893



In [8]:
model = NetMNIST()
model.load_state_dict(torch.load('train2_qat_sh7_9772.pt', map_location=torch.device('cpu')))

# Report (GetChannel(1))
for shift in (range(7,8)):
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_fxp(model, shift), test_loader)}\n")


layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear
Shift: 7, accuracy: 0.9770569801330566

